Comparing the face prediction success rate, given similar amounts of data obtained by TT-SVD and SVD.

- License-Identifier: GPL-3.0-only

- This file is part of the TT-sandbox project.

- Copyright © 2025 Idiap Research Institute <contact@idiap.ch>

- Contributor: Teng Xue <teng.xue@idiap.ch>

In [1]:
import pickle, sys, os
import numpy as np
from utils import tt_svd_full, tt_svd_rank, tt_svd_thres, tt_svd_recon
from utils import svd_full, svd_rank, svd_thres, svd_recon
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

## Data load

In [2]:
with open(os.getcwd()+'/data/facex.pickle', 'rb') as handle:
    x = pickle.load(handle)

with open(os.getcwd()+'/data/facey.pickle', 'rb') as handle:
    y = pickle.load(handle)

In [3]:
predictor = "SVC" #SVC or knn

## TT-SVD

In [4]:
x3d = x.reshape(x.shape[0], 168, 192) #for TT_SVD
x_train, x_test, y_train, y_test = train_test_split(x3d, y, test_size=0.1, random_state=42)
x_train_cores = tt_svd_thres(x_train, threshold=80)
x_train_recon = tt_svd_recon(x_train_cores).squeeze()
x_train_new =x_train_recon.reshape(x_train.shape[0], -1)

error = np.linalg.norm(x_train_recon - x_train)
print("The l2 norm error of TT-SVD is:", error)


num_elements = 0
for i in range(len(x_train_cores)):
    num_elements += x_train_cores[i].size
print("The numbers of elements in tt-svd is:", num_elements)

x_test_new =x_test.reshape(x_test.shape[0], -1)
if predictor=="SVC":
    # Train an SVM classifier on the decomposed data
    clf = SVC(kernel='linear')
    clf.fit(x_train_new, y_train)

    # Make predictions on the test set
    y_pred = clf.predict(x_test_new)
else:
    # Train a K-NN classifier on the decomposed data
    knn = KNeighborsClassifier(n_neighbors=5)  # You can adjust the number of neighbors (k) as needed
    knn.fit(x_train_new, y_train)
    # Make predictions on the test set
    y_pred = knn.predict(x_test_new)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy of TT_SVD: {accuracy:.2f}")

The ranks of truncated 0-th core is (1, 877, 30)
The ranks of truncated 1-th core is (30, 168, 16)
The ranks of truncated 2-th core is (16, 192, 1)
The l2 norm error of TT-SVD is: 537.4578677546946
The numbers of elements in tt-svd is: 110022
Accuracy of TT_SVD: 0.93


## Standard SVD

In [5]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=42)
U, S, V = svd_full(X=x_train)
assert V.shape[1]<num_elements, "Number of elements are too few to keep the shape of V matrix. Please increase ranks of TT_SVD!"
r_des = int(num_elements/(U.shape[0]+1+V.shape[1])) +1 #U.shape[0]*r + r + r*V.shape[1] = num_elements
U_new = U[:, :r_des]
S_new = S[:r_des]
V_new = V[:r_des, :]
# V[:, v_trun_dim:] = 0
print(f"U shape: {U_new.shape}, S shape: {S_new.shape}, V shape: {V_new.shape}")
svd_x_train = svd_recon(U_new, S_new, V_new).squeeze()
error = np.linalg.norm(x_train - svd_x_train)
print("The l2 norm error of standard SVD is:", error)
num_svd_elements = U_new.size + S_new.size + V_new.size
print("The numbers of elements in svd is:", num_svd_elements)

# face prediction
if predictor=="SVC":
    # Train an SVM classifier on the decomposed data
    clf = SVC(kernel='linear')
    clf.fit(svd_x_train, y_train)   

    # Make predictions on the test set
    y_pred = clf.predict(x_test)
else:
    # Train a K-NN classifier on the decomposed data
    knn = KNeighborsClassifier(n_neighbors=5)  # You can adjust the number of neighbors (k) as needed
    knn.fit(svd_x_train, y_train)
    # Make predictions on the test set
    y_pred = knn.predict(x_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy of standard SVD: {accuracy:.2f}")


U shape: (877, 4), S shape: (4,), V shape: (4, 32256)
The l2 norm error of standard SVD is: 894.1359037210595
The numbers of elements in svd is: 132536
Accuracy of standard SVD: 0.18
